# SuperMockLoad — quickstart

Load a SuperMock skypatch, look at the basics, and make a couple of plots.
The package ships the observational comparison data; point it at the synthetic
catalogs via `SUPERMOCK_DATA` or `root=`.

In [1]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
from supermockload import SuperMock, available_patches, plots, observations

# Set the data root once:
# import os; os.environ['SUPERMOCK_DATA'] = '/path/to/Mocks_v3_data'
# or pass root=... to SuperMock(...).

In [2]:
import os; os.environ['SUPERMOCK_DATA'] = '/lcrc/project/cosmo_ai/nramachandra/Projects/SPHEREx/Mocks_v3_data'

In [3]:
available_patches()

[]

### Load one patch
`downsample=` keeps a random subset (fast, low memory). Drop it for all rows.

In [4]:
sm = SuperMock(3, downsample=300_000)
sm

FileNotFoundError: /home/nramachandra/anaconda3/envs/env_jax_2024/lib/python3.9/site-packages/lightcone_catalogs/lightcone_galaxies_skypatch_3.h5

In [ ]:
print('footprint  :', round(sm.area_deg2, 1), 'deg^2')
print('N galaxies :', f'{sm.n:,}  (downsampled)')
print('z median   :', round(np.median(sm.redshift), 3))
print('logM* median (obs-epoch):', round(np.median(sm.logM_zobs), 2))

### Photometry & luminosities
Surveys: `LSST WISE SPHEREx COSMOS LEGACYSURVEY 2MASS F784` (all AB).

In [ ]:
mi = sm.mag('LSST', 'i')          # one band
print('LSST i median :', round(np.nanmedian(mi), 2))
print('SPHEREx matrix:', sm.survey('SPHEREx').shape, '| labels e.g.', sm.bands('SPHEREx')[:3])
print('M_r (rest)    :', round(np.nanmedian(sm.abs_mag('SDSS', 'r')), 2))

### A couple of plots vs observations

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
plots.redshift_distribution(sm, ax=ax[0])
plots.gsmf(sm, ax=ax[1])
plots.number_counts(sm, ax=ax[2], survey='LSST', band='i')
fig.tight_layout()

### Fast repeat loads: snapshots

The catalog/photometry files are gzip-compressed, so the first read is slow.
Snapshot a downsampled view to an uncompressed file that reloads instantly.

In [ ]:
rows = sm.sample(2000)                                  # for a stored SED subset
# needs seds=True to store SEDs; here we snapshot catalog+photometry only:
sm.save('patch3_300k.snapshot.h5', surveys=('LSST', 'WISE'))
sm2 = SuperMock.from_file('patch3_300k.snapshot.h5')    # instant
sm2